# BioSkills Lab — Chapter 11
## scVI and Single-Cell Representations
[![BioSkills Lab](https://img.shields.io/badge/BioSkills-Lab-38bdf8)](https://bioskillslab.dev)

> **GPU recommended** for faster scVI training

In [ ]:
!pip install -q scvi-tools scanpy anndata

In [ ]:
import scanpy as sc
import scvi
print(f'scanpy: {sc.__version__}  scvi: {scvi.__version__}')

In [ ]:
# Load PBMC 3k
adata = sc.datasets.pbmc3k()
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
adata = adata[adata.obs.pct_counts_mt < 5].copy()
sc.pp.highly_variable_genes(adata, n_top_genes=2000, subset=True, flavor='seurat_v3')
adata.layers['counts'] = adata.X.copy()
print(adata)

In [ ]:
# Train scVI
scvi.model.SCVI.setup_anndata(adata, layer='counts')
model = scvi.model.SCVI(adata, n_latent=20, n_layers=2)
model.train(max_epochs=200, early_stopping=True)
adata.obsm['X_scVI'] = model.get_latent_representation()
print('scVI training complete')

In [ ]:
# Cluster and visualise
sc.pp.neighbors(adata, use_rep='X_scVI')
sc.tl.leiden(adata, resolution=0.5)
sc.tl.umap(adata)
sc.pl.umap(adata, color=['leiden'], title='scVI clusters')

In [ ]:
# Compare with standard PCA pipeline
adata_pca = adata.copy()
sc.pp.normalize_total(adata_pca, target_sum=1e4)
sc.pp.log1p(adata_pca)
sc.pp.scale(adata_pca)
sc.tl.pca(adata_pca)
sc.pp.neighbors(adata_pca, n_pcs=40)
sc.tl.leiden(adata_pca, resolution=0.5, key_added='leiden_pca')
sc.tl.umap(adata_pca)

fig, axes = plt.subplots(1,2,figsize=(14,6))
sc.pl.umap(adata, color='leiden', ax=axes[0], show=False, title='scVI')
sc.pl.umap(adata_pca, color='leiden_pca', ax=axes[1], show=False, title='PCA')
plt.tight_layout(); plt.show()